In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1PV26OBTMbSeRzOOrt9s7XN5-1uPWDT8c", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# Notebook 2: Expert Parallelism Simulation -- Vizuara

In this notebook, we will simulate **expert parallelism** -- the technique of distributing MoE experts across multiple GPUs. We will implement the **All-to-All** communication pattern that makes this possible, measure its costs, and understand why expert parallelism requires fast interconnects.

Since we are working on a single GPU (Colab T4), we will simulate multi-GPU behavior using separate tensor partitions. The logic is identical to what real frameworks like Megatron-LM execute across actual GPUs.

**What you will build:**
- A simulated All-to-All communication primitive
- Expert parallelism dispatch and combine steps
- A complete forward pass with expert parallelism
- Communication cost analysis

**Estimated time:** 40--60 minutes

**Prerequisites:** Notebook 1 (MoE layer basics)

In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
import sys

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why All To All Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_all_to_all_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Why All-to-All?

In data parallelism, we used **AllReduce** -- every GPU sends the same type of data (gradients) and receives the same aggregated result. Expert parallelism is fundamentally different: each GPU needs to send **different tokens to different GPUs** based on routing decisions.

- GPU 0 has some tokens destined for experts on GPU 1, different tokens for GPU 2, etc.
- Simultaneously, GPU 0 receives tokens from all other GPUs destined for its local experts.

This is the **All-to-All** communication primitive. Let us build it.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Gpu Expert Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_simulated_gpu_expert_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimulatedGPU:
    """Represents one GPU in an expert-parallel group."""
    def __init__(self, gpu_id, local_experts, d_model):
        self.gpu_id = gpu_id
        self.local_experts = local_experts  # list of Expert modules
        self.num_local_experts = len(local_experts)
        self.d_model = d_model

        # Buffers for communication
        self.send_buffers = {}   # gpu_id -> tokens to send
        self.recv_buffers = {}   # gpu_id -> tokens received

    def __repr__(self):
        return f"GPU({self.gpu_id}, experts={self.num_local_experts})"


class Expert(nn.Module):
    """A single expert MLP."""
    def __init__(self, d_model, d_ff, expert_id):
        super().__init__()
        self.expert_id = expert_id
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.w2(self.act(self.w1(x)))


# Setup: 8 experts across 4 GPUs (2 experts per GPU)
d_model = 256
d_ff = 1024
num_experts = 8
num_gpus = 4
experts_per_gpu = num_experts // num_gpus

# Create all experts
all_experts = [Expert(d_model, d_ff, i) for i in range(num_experts)]

# Distribute experts across simulated GPUs
gpus = []
for g in range(num_gpus):
    start = g * experts_per_gpu
    end = start + experts_per_gpu
    local = all_experts[start:end]
    gpus.append(SimulatedGPU(g, local, d_model))

print(f"Configuration:")
print(f"  {num_experts} experts across {num_gpus} GPUs")
print(f"  {experts_per_gpu} experts per GPU")
for gpu in gpus:
    expert_ids = [e.expert_id for e in gpu.local_experts]
    print(f"  {gpu}: experts {expert_ids}")

In [ ]:
#@title 🎧 Listen: Router Dispatch Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_router_dispatch_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Router and Dispatch Assignment

First, each GPU runs the router on its local tokens to determine which experts (and therefore which GPUs) each token needs.

In [ ]:
#@title 🎧 Code Walkthrough: Router Implementation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_router_implementation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SharedRouter(nn.Module):
    """Router shared across all GPUs (same weights on every GPU)."""
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        """Returns routing decisions for each token."""
        logits = self.gate(x)                           # (T, num_experts)
        probs = F.softmax(logits, dim=-1)               # (T, num_experts)
        top_k_probs, top_k_indices = torch.topk(probs, self.top_k, dim=-1)
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        return top_k_weights, top_k_indices, probs


router = SharedRouter(d_model, num_experts, top_k=2)

# Simulate: each GPU has 64 tokens
tokens_per_gpu = 64
top_k = 2

# Generate local tokens for each GPU
torch.manual_seed(42)
local_tokens = {}
for g in range(num_gpus):
    local_tokens[g] = torch.randn(tokens_per_gpu, d_model)

# Each GPU runs the router on its tokens
routing_decisions = {}
for g in range(num_gpus):
    weights, indices, probs = router(local_tokens[g])
    routing_decisions[g] = {
        'weights': weights,    # (T, top_k)
        'indices': indices,    # (T, top_k)
        'probs': probs,        # (T, num_experts)
    }

# Show routing from GPU 0
print(f"GPU 0 routing decisions (first 5 tokens):")
for t in range(5):
    idx = routing_decisions[0]['indices'][t].tolist()
    w = routing_decisions[0]['weights'][t].tolist()
    dest_gpus = [i // experts_per_gpu for i in idx]
    print(f"  Token {t}: experts={idx}, weights=[{w[0]:.3f}, {w[1]:.3f}], "
          f"dest GPUs={dest_gpus}")

In [ ]:
#@title 🎧 Listen: All To All Dispatch Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_all_to_all_dispatch_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Implementing All-to-All Dispatch

The All-to-All dispatch step sorts tokens by their destination GPU and "sends" them. In real multi-GPU code, this would use `torch.distributed.all_to_all`. Here, we simulate it.

In [ ]:
#@title 🎧 Code Walkthrough: All To All Dispatch Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_all_to_all_dispatch_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def all_to_all_dispatch(local_tokens, routing_decisions, num_gpus, experts_per_gpu):
    """
    Simulate All-to-All dispatch: route tokens from source GPUs to destination GPUs.

    Returns:
        recv_buffers: dict[gpu_id -> dict[expert_local_id -> (tokens, weights, source_info)]]
    """
    # For each GPU, prepare send buffers partitioned by destination
    recv_buffers = {g: {} for g in range(num_gpus)}

    for src_gpu in range(num_gpus):
        tokens = local_tokens[src_gpu]
        indices = routing_decisions[src_gpu]['indices']  # (T, top_k)
        weights = routing_decisions[src_gpu]['weights']  # (T, top_k)
        T, K = indices.shape

        for t in range(T):
            for k in range(K):
                expert_id = indices[t, k].item()
                dest_gpu = expert_id // experts_per_gpu
                local_expert_id = expert_id % experts_per_gpu
                weight = weights[t, k].item()

                # "Send" token to destination GPU
                if local_expert_id not in recv_buffers[dest_gpu]:
                    recv_buffers[dest_gpu][local_expert_id] = []
                recv_buffers[dest_gpu][local_expert_id].append({
                    'token': tokens[t],
                    'weight': weight,
                    'src_gpu': src_gpu,
                    'src_token_idx': t,
                    'k_idx': k,
                })

    return recv_buffers


recv_buffers = all_to_all_dispatch(
    local_tokens, routing_decisions, num_gpus, experts_per_gpu
)

# Analyze the dispatch
print("All-to-All Dispatch Results:")
print("=" * 60)

total_transfers = 0
cross_gpu_transfers = 0

for gpu_id in range(num_gpus):
    total_tokens = sum(len(v) for v in recv_buffers[gpu_id].values())
    from_self = 0
    from_others = 0
    for expert_id, tokens_list in recv_buffers[gpu_id].items():
        for tok in tokens_list:
            if tok['src_gpu'] == gpu_id:
                from_self += 1
            else:
                from_others += 1
    total_transfers += total_tokens
    cross_gpu_transfers += from_others
    print(f"GPU {gpu_id}: received {total_tokens} tokens "
          f"({from_self} local, {from_others} from other GPUs)")
    for expert_id in sorted(recv_buffers[gpu_id].keys()):
        print(f"  Expert {gpu_id * experts_per_gpu + expert_id}: "
              f"{len(recv_buffers[gpu_id][expert_id])} tokens")

print(f"\nTotal token-expert assignments: {total_transfers}")
print(f"Cross-GPU transfers: {cross_gpu_transfers} "
      f"({cross_gpu_transfers/total_transfers*100:.1f}%)")

In [ ]:
#@title 🎧 Listen: Expert Comp Combine Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_expert_comp_combine_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Expert Computation + All-to-All Combine

After dispatch, each GPU processes the received tokens with its local experts. Then a second All-to-All sends the results back.

In [ ]:
#@title 🎧 Code Walkthrough: Expert Comp Combine Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_expert_comp_combine_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def expert_compute_and_combine(gpus, recv_buffers, local_tokens,
                                routing_decisions, num_gpus, experts_per_gpu):
    """
    Step 3-5: Expert computation + All-to-All combine + weighted sum.

    Returns:
        outputs: dict[gpu_id -> (tokens_per_gpu, d_model)] final outputs
    """
    tokens_per_gpu = local_tokens[0].shape[0]
    d_model = local_tokens[0].shape[1]

    # Initialize output buffers
    outputs = {g: torch.zeros(tokens_per_gpu, d_model) for g in range(num_gpus)}

    # Process on each GPU
    for gpu_id in range(num_gpus):
        for local_expert_id, token_list in recv_buffers[gpu_id].items():
            expert = gpus[gpu_id].local_experts[local_expert_id]

            # Batch the tokens for this expert
            if len(token_list) == 0:
                continue

            token_batch = torch.stack([t['token'] for t in token_list])

            # Run expert
            with torch.no_grad():
                expert_outputs = expert(token_batch)

            # "Send back" results (All-to-All combine)
            for i, tok_info in enumerate(token_list):
                src_gpu = tok_info['src_gpu']
                src_idx = tok_info['src_token_idx']
                weight = tok_info['weight']
                # Weighted addition to output
                outputs[src_gpu][src_idx] += weight * expert_outputs[i]

    return outputs


# Run the full expert parallelism forward pass
final_outputs = expert_compute_and_combine(
    gpus, recv_buffers, local_tokens, routing_decisions,
    num_gpus, experts_per_gpu
)

print("Expert Parallelism Forward Pass Complete!")
for g in range(num_gpus):
    norm = final_outputs[g].norm(dim=-1).mean().item()
    print(f"  GPU {g} output norm (mean): {norm:.4f}")

In [ ]:
#@title 🎧 Listen: Visualize Comm Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_visualize_comm_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Visualizing the Communication Pattern

Let us visualize the All-to-All pattern to see which GPUs send tokens to which.

In [ ]:
#@title 🎧 What to Look For: Comm Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_comm_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Build a communication matrix: comm[src][dst] = number of tokens sent
comm_matrix = np.zeros((num_gpus, num_gpus))

for dst_gpu in range(num_gpus):
    for expert_id, token_list in recv_buffers[dst_gpu].items():
        for tok in token_list:
            comm_matrix[tok['src_gpu']][dst_gpu] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Communication matrix heatmap
im = axes[0].imshow(comm_matrix, cmap='YlOrRd')
axes[0].set_xlabel('Destination GPU', fontsize=12)
axes[0].set_ylabel('Source GPU', fontsize=12)
axes[0].set_title('All-to-All Communication Matrix\n(tokens sent from src to dst)', fontsize=13)
axes[0].set_xticks(range(num_gpus))
axes[0].set_yticks(range(num_gpus))
for i in range(num_gpus):
    for j in range(num_gpus):
        color = 'white' if comm_matrix[i][j] > comm_matrix.max() * 0.6 else 'black'
        axes[0].text(j, i, f"{int(comm_matrix[i][j])}",
                     ha='center', va='center', color=color, fontsize=14)
plt.colorbar(im, ax=axes[0], label='Tokens')

# Per-GPU send/receive totals
sent = comm_matrix.sum(axis=1)
received = comm_matrix.sum(axis=0)
local = np.diag(comm_matrix)

x_pos = np.arange(num_gpus)
width = 0.25
axes[1].bar(x_pos - width, sent, width, label='Sent', color='#2196F3')
axes[1].bar(x_pos, received, width, label='Received', color='#4CAF50')
axes[1].bar(x_pos + width, local, width, label='Local (no transfer)', color='#FFC107')
axes[1].set_xlabel('GPU', fontsize=12)
axes[1].set_ylabel('Tokens', fontsize=12)
axes[1].set_title('Communication Volume per GPU', fontsize=13)
axes[1].set_xticks(x_pos)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Comm Cost Analysis Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_comm_cost_analysis_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Communication Cost Analysis

Let us compute the actual data transfer volumes for realistic model configurations.

$$\text{Data per GPU} = T \cdot d \cdot k \cdot \text{sizeof(float)}$$

In [ ]:
#@title 🎧 Code Walkthrough: Comm Cost Calculation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_comm_cost_calculation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_all_to_all_cost(tokens_per_gpu, d_model, top_k, num_moe_layers,
                             dtype_bytes=2):
    """
    Compute All-to-All communication cost.

    Args:
        tokens_per_gpu: number of tokens on each GPU
        d_model: hidden dimension
        top_k: number of experts per token
        num_moe_layers: number of MoE layers in the model
        dtype_bytes: bytes per element (2 for FP16/BF16, 4 for FP32)

    Returns:
        dict with communication costs
    """
    # Per MoE layer: each GPU sends and receives tokens
    # In the worst case, ALL tokens need to go to OTHER GPUs
    bytes_per_layer = tokens_per_gpu * d_model * top_k * dtype_bytes

    # Two All-to-All per layer (dispatch + combine)
    bytes_per_layer_total = 2 * bytes_per_layer

    # Total across all MoE layers
    total_bytes = bytes_per_layer_total * num_moe_layers

    return {
        'per_layer_MB': bytes_per_layer_total / 1e6,
        'total_MB': total_bytes / 1e6,
        'total_GB': total_bytes / 1e9,
    }


# Realistic model configurations
configs = [
    {'name': 'Mixtral 8x7B', 'T': 2048, 'd': 4096, 'k': 2, 'layers': 32},
    {'name': 'DeepSeek-V3', 'T': 4096, 'd': 7168, 'k': 8, 'layers': 60},
    {'name': 'Switch-C', 'T': 2048, 'd': 1536, 'k': 1, 'layers': 64},
]

print(f"{'Model':<20} {'Per Layer (MB)':>15} {'Total (GB)':>12} {'Layers':>8}")
print("=" * 60)
for c in configs:
    cost = compute_all_to_all_cost(c['T'], c['d'], c['k'], c['layers'])
    print(f"{c['name']:<20} {cost['per_layer_MB']:>15.1f} {cost['total_GB']:>12.2f} {c['layers']:>8}")

print("\nNote: These are upper-bound estimates (worst-case where all tokens")
print("cross GPU boundaries). Actual volume depends on routing decisions.")

# Compare with NVLink bandwidth
nvlink_bw = 900  # GB/s for NVLink 4.0 bidirectional
print(f"\nNVLink 4.0 bandwidth: {nvlink_bw} GB/s")
for c in configs:
    cost = compute_all_to_all_cost(c['T'], c['d'], c['k'], c['layers'])
    time_ms = cost['total_GB'] / nvlink_bw * 1000
    print(f"  {c['name']}: {cost['total_GB']:.2f} GB takes ~{time_ms:.2f} ms on NVLink")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Your Turn -- TODO Exercises

### TODO 1: Implement the Complete Expert Parallel Forward Pass

Combine all the steps into a single function that takes local tokens and returns outputs.

In [ ]:
def expert_parallel_forward(local_tokens_dict, router, gpus,
                             num_gpus, experts_per_gpu):
    """
    Complete expert parallelism forward pass.

    Steps:
      1. Route locally on each GPU
      2. All-to-All dispatch
      3. Expert computation
      4. All-to-All combine
      5. Weighted summation

    Args:
        local_tokens_dict: dict[gpu_id -> (T, d_model)]
        router: SharedRouter
        gpus: list of SimulatedGPU
        num_gpus: int
        experts_per_gpu: int

    Returns:
        outputs: dict[gpu_id -> (T, d_model)]
        stats: dict with routing statistics
    """
    # ============ TODO ============
    # Step 1: Run router on each GPU's local tokens
    routing_decisions = {}
    for g in range(num_gpus):
        weights, indices, probs = ???  # YOUR CODE HERE
        routing_decisions[g] = {
            'weights': weights,
            'indices': indices,
            'probs': probs,
        }

    # Step 2: All-to-All dispatch
    recv_buffers = all_to_all_dispatch(
        ???  # YOUR CODE HERE -- pass the right arguments
    )

    # Steps 3-5: Expert compute + combine
    outputs = expert_compute_and_combine(
        ???  # YOUR CODE HERE -- pass the right arguments
    )
    # ==============================


In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo_1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    # Compute stats
    total_tokens = sum(t.shape[0] for t in local_tokens_dict.values())
    cross_gpu = 0
    total_assignments = 0
    for dst in range(num_gpus):
        for eid, tlist in recv_buffers[dst].items():
            for tok in tlist:
                total_assignments += 1
                if tok['src_gpu'] != dst:
                    cross_gpu += 1

    stats = {
        'total_tokens': total_tokens,
        'total_assignments': total_assignments,
        'cross_gpu_transfers': cross_gpu,
        'cross_gpu_pct': cross_gpu / total_assignments * 100,
    }

    return outputs, stats


# Test
torch.manual_seed(42)
test_tokens = {g: torch.randn(32, d_model) for g in range(num_gpus)}

outputs, stats = expert_parallel_forward(
    test_tokens, router, gpus, num_gpus, experts_per_gpu
)

print("Expert Parallel Forward Pass:")
print(f"  Total tokens: {stats['total_tokens']}")
print(f"  Total assignments: {stats['total_assignments']}")
print(f"  Cross-GPU transfers: {stats['cross_gpu_transfers']} ({stats['cross_gpu_pct']:.1f}%)")

for g in range(num_gpus):
    assert outputs[g].shape == test_tokens[g].shape, \
        f"GPU {g}: output shape {outputs[g].shape} != input shape {test_tokens[g].shape}"
print("\nTODO 1 passed! All output shapes match input shapes.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Measure Expert Load Distribution Across GPUs

Compute how many tokens each GPU processes. Identify the most and least loaded GPUs.

In [ ]:
def measure_gpu_load(recv_buffers, num_gpus):
    """
    Measure the computational load on each GPU.

    Args:
        recv_buffers: output from all_to_all_dispatch
        num_gpus: int

    Returns:
        loads: list of ints -- number of tokens processed per GPU
        imbalance_ratio: float -- max_load / min_load
    """
    # ============ TODO ============
    loads = []
    for g in range(num_gpus):
        # Count total tokens received by this GPU (across all its experts)
        gpu_load = ???  # YOUR CODE HERE
        loads.append(gpu_load)

    imbalance_ratio = ???  # YOUR CODE HERE: max(loads) / min(loads)
    # ==============================


In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo_2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    return loads, imbalance_ratio


# Use the recv_buffers from the earlier dispatch
loads, imbalance = measure_gpu_load(recv_buffers, num_gpus)

print(f"GPU loads: {loads}")
print(f"Imbalance ratio (max/min): {imbalance:.2f}")
print(f"Ideal load per GPU: {tokens_per_gpu * top_k / num_gpus * num_gpus:.0f} tokens")

# Visualize
plt.figure(figsize=(8, 4))
colors = ['#f44336' if l == max(loads) else '#4CAF50' if l == min(loads)
          else '#2196F3' for l in loads]
plt.bar(range(num_gpus), loads, color=colors, edgecolor='white')
ideal = sum(loads) / num_gpus
plt.axhline(y=ideal, color='gray', linestyle='--', label=f'Ideal: {ideal:.0f}')
plt.xlabel('GPU', fontsize=12)
plt.ylabel('Tokens to Process', fontsize=12)
plt.title('GPU Load Distribution in Expert Parallelism', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Vary Number of GPUs and Measure Communication

How does cross-GPU communication scale as we increase the number of GPUs? More GPUs means each GPU has fewer local experts, so more tokens need to cross GPU boundaries.

In [ ]:
# ============ TODO ============
# For each GPU count in [1, 2, 4, 8], create the expert-parallel setup
# and measure the fraction of cross-GPU token transfers.
#
# Hint: reuse the all_to_all_dispatch function and count cross-GPU tokens.

gpu_counts = [1, 2, 4, 8]
cross_gpu_fractions = []

for ng in gpu_counts:
    epg = num_experts // ng  # experts per GPU

    # Create experts and GPUs
    test_experts = [Expert(d_model, d_ff, i) for i in range(num_experts)]
    test_gpus = []
    for g in range(ng):
        local = test_experts[g*epg:(g+1)*epg]
        test_gpus.append(SimulatedGPU(g, local, d_model))

    # Generate tokens and route
    torch.manual_seed(42)
    tpg = 256 // ng  # Keep total tokens constant at 256
    test_local = {g: torch.randn(tpg, d_model) for g in range(ng)}
    test_routing = {}
    for g in range(ng):
        w, idx, p = router(test_local[g])
        test_routing[g] = {'weights': w, 'indices': idx, 'probs': p}

    # Dispatch and count
    test_recv = all_to_all_dispatch(test_local, test_routing, ng, epg)

    # YOUR CODE HERE: count cross-GPU fraction
    cross = 0
    total = 0
    for dst in range(ng):
        for eid, tlist in test_recv[dst].items():
            for tok in tlist:
                total += 1
                if tok['src_gpu'] != dst:
                    cross += 1

    frac = ???  # YOUR CODE HERE: cross / total * 100


In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo_3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    cross_gpu_fractions.append(frac)
# ==============================

# Plot
plt.figure(figsize=(8, 5))
plt.plot(gpu_counts, cross_gpu_fractions, 'o-', color='#f44336',
         linewidth=2, markersize=10)
plt.xlabel('Number of GPUs (Expert Parallelism Degree)', fontsize=12)
plt.ylabel('Cross-GPU Token Transfers (%)', fontsize=12)
plt.title('Communication Cost vs Expert Parallelism Degree', fontsize=14)
plt.xticks(gpu_counts)
plt.grid(True, alpha=0.3)

# Add theoretical line: (1 - 1/N) * 100
theoretical = [(1 - 1/ng) * 100 for ng in gpu_counts]
plt.plot(gpu_counts, theoretical, '--', color='gray',
         label='Theoretical: (1 - 1/N) * 100%')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nResults:")
for ng, frac in zip(gpu_counts, cross_gpu_fractions):
    print(f"  {ng} GPUs: {frac:.1f}% cross-GPU transfers")
print("\nMore GPUs = more communication. This is why EP is usually 4-16 GPUs.")

In [ ]:
#@title 🎧 Listen: All To All Vs Allreduce Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_all_to_all_vs_allreduce_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Comparing All-to-All vs AllReduce

Let us contrast the two communication patterns.

In [ ]:
#@title 🎧 Code Walkthrough: Comparison Details Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_comparison_details_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Conceptual comparison: AllReduce vs All-to-All
print("AllReduce (Data Parallelism):")
print("  - Every GPU sends the SAME data (gradients) to all others")
print("  - Every GPU receives the SAME result (averaged gradients)")
print("  - Cost: O(M) where M = model size (independent of GPU count with ring)")
print("  - Scales well to many GPUs")
print()
print("All-to-All (Expert Parallelism):")
print("  - Every GPU sends DIFFERENT data to DIFFERENT GPUs")
print("  - Data routing depends on learned router decisions")
print("  - Cost: O(T * d * k) per MoE layer")
print("  - Two All-to-All per layer (dispatch + combine)")
print("  - Needs fast interconnect (NVLink/NVSwitch)")
print()

# Quantitative comparison for a concrete model
d = 4096
T_local = 2048
k_top = 2
model_params = 47e9  # Mixtral 8x7B
moe_layers = 32

# AllReduce: gradients for the full model
allreduce_bytes = model_params * 4  # FP32 gradients
print(f"AllReduce data volume: {allreduce_bytes / 1e9:.1f} GB (once per step)")

# All-to-All: per MoE layer
a2a_per_layer = 2 * T_local * d * k_top * 2  # 2 transfers, FP16
a2a_total = a2a_per_layer * moe_layers
print(f"All-to-All data volume: {a2a_total / 1e9:.2f} GB ({moe_layers} layers)")
print(f"  Per layer: {a2a_per_layer / 1e6:.1f} MB")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### What We Built
- A simulated **multi-GPU expert parallelism** system
- The **All-to-All** dispatch and combine communication pattern
- Communication cost analysis for realistic model configurations

### Key Takeaways
1. All-to-All sends **different data to different GPUs** -- fundamentally different from AllReduce
2. More expert-parallel GPUs means more cross-GPU communication
3. Communication cost scales with $T \cdot d \cdot k$ per MoE layer
4. Expert parallelism typically stays within 4--16 GPUs on fast interconnects

### What Comes Next
In Notebook 3, we will implement **load balancing** (the auxiliary loss) and explore how expert parallelism fits into the **5D parallelism grid**.